In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\AI-Powered-Intelligent-Traffic-Management-System\datasets\processed\traffic_ml_dataset.csv"
)

df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,year,month,day,hour,day_of_week,week,is_weekend,is_peak_hour
0,3,288.28,0.0,0.0,40,1,24,2012-10-02 09:00:00,5545,2012,10,2,9,1,40,0,1
1,3,289.36,0.0,0.0,75,1,2,2012-10-02 10:00:00,4516,2012,10,2,10,1,40,0,1
2,3,289.58,0.0,0.0,90,1,19,2012-10-02 11:00:00,4767,2012,10,2,11,1,40,0,0
3,3,290.13,0.0,0.0,90,1,19,2012-10-02 12:00:00,5026,2012,10,2,12,1,40,0,0
4,3,291.14,0.0,0.0,75,1,2,2012-10-02 13:00:00,4918,2012,10,2,13,1,40,0,0


In [2]:
df["date_time"] = pd.to_datetime(df["date_time"])

In [3]:
df["year"] = df["date_time"].dt.year

df["month"] = df["date_time"].dt.month

df["day"] = df["date_time"].dt.day

df["hour"] = df["date_time"].dt.hour

df["minute"] = df["date_time"].dt.minute

df["day_of_week"] = df["date_time"].dt.dayofweek

df["week_of_year"] = df["date_time"].dt.isocalendar().week.astype(int)

df["quarter"] = df["date_time"].dt.quarter

In [4]:
df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

In [5]:
df["is_peak_hour"] = (
    (
        (df["hour"] >= 7) &
        (df["hour"] <= 10)
    )
    |
    (
        (df["hour"] >= 16) &
        (df["hour"] <= 19)
    )
).astype(int)

In [6]:
df["is_night"] = (
    (
        df["hour"] >= 22
    )
    |
    (
        df["hour"] <= 5
    )
).astype(int)

In [7]:
def rush_hour(hour):

    if 7 <= hour <= 10:
        return "Morning"

    elif 16 <= hour <= 19:
        return "Evening"

    else:
        return "Normal"

df["rush_hour"] = df["hour"].apply(rush_hour)

In [8]:
df["weather_score"] = (

    df["rain_1h"] * 2 +

    df["snow_1h"] * 3 +

    df["clouds_all"] / 20

)

In [9]:
def temperature_category(temp):

    if temp < 270:
        return "Cold"

    elif temp < 295:
        return "Moderate"

    else:
        return "Hot"

df["temperature_category"] = (
    df["temp"]
    .apply(temperature_category)
)

In [10]:
df["traffic_previous_hour"] = (
    df["traffic_volume"]
    .shift(1)
)

In [11]:
df["traffic_rolling_mean"] = (

    df["traffic_volume"]

    .rolling(window=3)

    .mean()

)

In [12]:
df["traffic_rolling_max"] = (

    df["traffic_volume"]

    .rolling(3)

    .max()

)

In [13]:
df["traffic_rolling_min"] = (

    df["traffic_volume"]

    .rolling(3)

    .min()

)

In [14]:
df["traffic_std"] = (

    df["traffic_volume"]

    .rolling(3)

    .std()

)

In [15]:
df["traffic_change"] = (

    df["traffic_volume"]

    .pct_change()

)

In [16]:
df["traffic_growth"] = (

    df["traffic_volume"]

    .diff()

)

In [17]:
df = df.bfill()

In [18]:
q1 = df["traffic_volume"].quantile(0.25)

q2 = df["traffic_volume"].quantile(0.50)

q3 = df["traffic_volume"].quantile(0.75)


def congestion(volume):

    if volume <= q1:
        return "Low"

    elif volume <= q2:
        return "Medium"

    elif volume <= q3:
        return "High"

    return "Severe"

df["congestion"] = (
    df["traffic_volume"]
    .apply(congestion)
)

In [19]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["rush_hour"] = encoder.fit_transform(
    df["rush_hour"]
)

df["temperature_category"] = encoder.fit_transform(
    df["temperature_category"]
)

df["congestion"] = encoder.fit_transform(
    df["congestion"]
)

In [20]:
print("=" * 60)
print("Final Dataset Shape")
print(df.shape)

print("=" * 60)
print(df.info())

df.head()

Final Dataset Shape
(48187, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48187 entries, 0 to 48186
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   holiday                48187 non-null  int64         
 1   temp                   48187 non-null  float64       
 2   rain_1h                48187 non-null  float64       
 3   snow_1h                48187 non-null  float64       
 4   clouds_all             48187 non-null  int64         
 5   weather_main           48187 non-null  int64         
 6   weather_description    48187 non-null  int64         
 7   date_time              48187 non-null  datetime64[ns]
 8   traffic_volume         48187 non-null  int64         
 9   year                   48187 non-null  int32         
 10  month                  48187 non-null  int32         
 11  day                    48187 non-null  int32         
 12  hour                   48187

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,year,...,weather_score,temperature_category,traffic_previous_hour,traffic_rolling_mean,traffic_rolling_max,traffic_rolling_min,traffic_std,traffic_change,traffic_growth,congestion
0,3,288.28,0.0,0.0,40,1,24,2012-10-02 09:00:00,5545,2012,...,2.00,2,5545.0,4942.666667,5545.0,4516.0,536.520581,-0.185573,-1029.0,3
1,3,289.36,0.0,0.0,75,1,2,2012-10-02 10:00:00,4516,2012,...,3.75,2,5545.0,4942.666667,5545.0,4516.0,536.520581,-0.185573,-1029.0,0
2,3,289.58,0.0,0.0,90,1,19,2012-10-02 11:00:00,4767,2012,...,4.50,2,4516.0,4942.666667,5545.0,4516.0,536.520581,0.055580,251.0,0
3,3,290.13,0.0,0.0,90,1,19,2012-10-02 12:00:00,5026,2012,...,4.50,2,4767.0,4769.666667,5026.0,4516.0,255.010457,0.054332,259.0,3
4,3,291.14,0.0,0.0,75,1,2,2012-10-02 13:00:00,4918,2012,...,3.75,2,5026.0,4903.666667,5026.0,4767.0,130.093556,-0.021488,-108.0,0


In [21]:
df.to_csv(
    r"C:\AI-Powered-Intelligent-Traffic-Management-System\datasets\processed\traffic_feature_engineered.csv",
    index=False
)

print("Feature engineered dataset saved successfully.")

Feature engineered dataset saved successfully.
